In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q soundfile librosa
!pip install -q deep-filter
!pip install -q pesq pystoi
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载DeepFilterNet代码
import os
if not os.path.exists('DeepFilterNet-main'):
    print('正在克隆DeepFilterNet仓库...')
    !git clone https://github.com/Rikorose/DeepFilterNet.git DeepFilterNet-main
    print('克隆完成！')

# 下载预训练模型
model_dir = 'DeepFilterNet-main/models'
if not os.path.exists(os.path.join(model_dir, 'DeepFilterNet3')):
    print('正在下载预训练模型...')
    !cd DeepFilterNet-main/models && unzip -q DeepFilterNet3.zip 2>/dev/null || echo '需要手动上传模型权重'

# 生成测试音频
print('环境准备就绪！')


# Module 5 Session 2: DeepFilterNet Code Analysis + Running InferenceThis notebook covers:1. DeepFilterNet code structure overview2. Configuration and parameter analysis3. Core module deep dive with data flow tracking4. Comparison with DeepACE architecture5. Running inference with pretrained models6. Output analysis and visualization---

## Section 1: Code Structure OverviewDeepFilterNet-main directory structure:```DeepFilterNet-main/+-- DeepFilterNet/df/        # Python package (core code)|   +-- deepfilternet.py     # DfNet model (Encoder, ErbDecoder, DfDecoder)|   +-- deepfilternet2.py    # DeepFilterNet2 variant|   +-- deepfilternet3.py    # DeepFilterNet3 variant|   +-- modules.py           # Building blocks (Conv, GRU, Mask, DfOp)|   +-- config.py            # Configuration system (DfParams)|   +-- enhance.py           # Inference entry point|   +-- loss.py              # Loss functions (MaskLoss, SpectralLoss, etc.)|   +-- model.py             # Model parameter definitions|   +-- train.py             # Training script|   +-- io.py                # Audio I/O utilities|   +-- utils.py             # General utilities+-- models/                  # Pretrained model weights (.zip)+-- assets/                  # Sample audio files+-- libDF/                   # Rust core (STFT, ERB, data loading)+-- pyDF/                    # Python bindings for Rust core```> "See the forest first, then the trees" -- understand the overall architecture before diving into details.

In [ ]:
# Load configuration and understand parameters# DeepFilterNet uses a custom config system# Here we show the key default parametersparams = {    "SR": 48000,         # Sample rate    "FFT_SIZE": 960,     # FFT window size    "HOP_SIZE": 480,     # STFT hop size (50% overlap)    "NB_ERB": 32,        # Number of ERB bands    "NB_DF": 96,         # Number of deep filtering bins    "DF_ORDER": 5,       # Deep filtering order (temporal context)    "DF_LOOKAHEAD": 0,   # Lookahead frames for DF    "LSNR_MAX": 35,      # Max local SNR    "LSNR_MIN": -15,     # Min local SNR}print("=== DeepFilterNet Key Parameters ===")for k, v in params.items():    print("  %s: %s" % (k, v))# Derived valuessr = params["SR"]fft_size = params["FFT_SIZE"]hop_size = params["HOP_SIZE"]freq_bins = fft_size // 2 + 1print("\nDerived values:")print("  Frequency bins: %d" % freq_bins)print("  Frame duration: %.1f ms" % (fft_size / sr * 1000))print("  Hop duration: %.1f ms" % (hop_size / sr * 1000))print("  Frames per second: %.0f" % (sr / hop_size))

## Section 2: Model Architecture Deep Dive### DfNet Architecture (from deepfilternet.py)The main model class `DfNet` has 4 core components:1. **Encoder**: Processes ERB features and spectral features in parallel   - ERB path: 4 Conv2d layers with increasing channels   - DF path: 2 Conv2d layers for spectral features   - Both paths merge via GroupedGRU for temporal modeling2. **ErbDecoder**: U-Net style decoder with skip connections   - Reconstructs ERB-domain mask from encoded features   - Output: mask in range [0, 1] via Sigmoid3. **DfDecoder**: Generates Deep Filtering coefficients   - Takes encoded embedding and spectral features   - Output: DF coefficients (order x bins) + alpha (blending weight)4. **Mask + DfOp**: Apply enhancement   - Mask applied to full spectrogram via ERB inverse filterbank   - DfOp applies learned temporal filtering on low frequencies

In [ ]:
# Data flow analysis with tensor shapes# Based on DeepFilterNet3 configbatch = 2T = 100  # time framesF = freq_bins  # 481Fe = params["NB_ERB"]  # 32 ERB bandsFc = params["NB_DF"]  # 96 DF binsprint("=== DeepFilterNet Data Flow ===")print()print("Input features:")print("  feat_erb:  (%d, 1, %d, %d)  [batch, 1, T, ERB_bands]" % (batch, T, Fe))print("  feat_spec: (%d, 2, %d, %d)  [batch, 2(re/im), T, DF_bins]" % (batch, T, Fc))print("  spec:      (%d, 1, %d, %d, 2) [batch, 1, T, freq_bins, re/im]" % (batch, T, F))print()# Encoderprint("Encoder:")print("  e0: (B, 16, T, Fe)     # ERB Conv0")print("  e1: (B, 32, T, Fe/2)   # ERB Conv1")print("  e2: (B, 64, T, Fe/4)   # ERB Conv2")print("  e3: (B, 64, T, Fe/4)   # ERB Conv3")print("  c0: (B, 16, T, Fc)     # DF Conv0")print("  c1: (B, 32, T, Fc)     # DF Conv1")print("  emb: (B, T, hidden)     # Merged embedding via GroupedGRU")print("  lsnr: (B, T, 1)        # Local SNR estimate")print()# ErbDecoderprint("ErbDecoder (U-Net decoder):")print("  e3 + emb -> convt3 -> (B, 64, T, Fe/4)")print("  e2 + up   -> convt2 -> (B, 32, T, Fe/2)")print("  e1 + up   -> convt1 -> (B, 16, T, Fe)")print("  e0 + up   -> conv0  -> (B, 1, T, Fe)")print("  mask = Sigmoid(output)  # [0, 1] mask")print()# DfDecoderprint("DfDecoder:")print("  df_coefs: (B, T, O, F, 2)  # O=5 DF order, F=96 bins, re/im")print("  df_alpha: (B, T, 1)        # Blending weight")print()# Final outputprint("Final Output:")print("  enhanced_spec: (B, 1, T, F, 2)  # Enhanced complex spectrogram")

## Section 3: Key Modules Analysis### GroupedGRU- Splits GRU into G independent groups- Each group processes a subset of features- Reduces parameters while maintaining capacity- Similar to grouped convolution idea from Module 2### Mask Module- Converts ERB-domain mask back to full frequency resolution- Uses ERB inverse filterbank: mask_full = mask_erb @ erb_inv_fb- Applies mask multiplicatively: enhanced = spec * mask### DfOp (Deep Filtering Operation)- Learned temporal FIR filtering on low-frequency bins- Y[f] = sum_{k=0}^{K-1} C[k,f] * X[f-k]  (complex multiplication)- alpha controls blending: output = alpha * filtered + (1-alpha) * masked- 5 different implementation variants for different deployment targets

## Section 4: Comparison with DeepACE| Aspect | DeepACE (Module 4) | DeepFilterNet (Module 5) ||--------|-------------------|------------------------|| Task | Electrodogram prediction | Speech enhancement || Input | Noisy speech waveform | Noisy speech spectrogram || Output | 22-channel electrodogram | Enhanced speech spectrogram || Domain | Time domain (Conv1d) | Time-frequency domain (Conv2d) || Encoder | Conv1d (learned features) | Conv2d on ERB features || Temporal model | Dilated Conv + SE | GroupedGRU || Decoder | ConvTranspose1d | U-Net with skip connections || Key innovation | ChannelRebalancer for CI | Deep Filtering stage || Sampling rate | 16 kHz | 48 kHz || Real-time | Yes (causal) | Yes (~5ms latency) |> Both models share the encoder-decoder-mask pattern but apply it to different domains and for different purposes.

## Section 5: Running InferenceDeepFilterNet provides pretrained models that can process noisy audio directly.There are several ways to run inference:1. Command line: `python -m df.enhance <input_dir>`2. Python API: using the `enhance()` function3. Pip package: `deep-filter` commandFor this notebook, we will use the Python API approach.

In [ ]:
# Running DeepFilterNet inferenceimport osimport sysDFN_DIR = os.path.join('DeepFilterNet-main', 'DeepFilterNet')sys.path.insert(0, DFN_DIR)# Check if DeepFilterNet is installed/importabletry:    from df.config import config    config.use_defaults()    from df.model import ModelParams    p = ModelParams()    print("DeepFilterNet config loaded!")    print("  SR: %d" % p.sr)    print("  FFT_SIZE: %d" % p.fft_size)    print("  HOP_SIZE: %d" % p.hop_size)    print("  NB_ERB: %d" % p.nb_erb)    print("  NB_DF: %d" % p.nb_df)    print("  DF_ORDER: %d" % p.df_order)    DF_AVAILABLE = Trueexcept ImportError as e:    print("DeepFilterNet not importable:", e)    print("Install with: pip install deep-filter")    print("Or: cd %s && pip install -e ." % DFN_DIR)    DF_AVAILABLE = False

In [ ]:
# Load pretrained model and run inferenceif DF_AVAILABLE:    try:        from df.enhance import init_df, enhance        from df.io import load_audio, save_audio        model, df_state, suffix, epoch = init_df(            os.path.join('DeepFilterNet-main', 'models', 'DeepFilterNet3'),            post_filter=True,            config_allow_defaults=True,        )        print("Pretrained model loaded! (epoch %d)" % epoch)        # Check for test audio        test_dir = os.path.join('test_samples')        assets_dir = os.path.join('DeepFilterNet-main', 'assets')        test_files = []        if os.path.exists(test_dir):            for f in sorted(os.listdir(test_dir)):                if f.endswith('.wav'):                    test_files.append(os.path.join(test_dir, f))        if not test_files and os.path.exists(assets_dir):            noisy_file = os.path.join(assets_dir, 'noisy_snr0.wav')            if os.path.exists(noisy_file):                test_files.append(noisy_file)        if test_files:            print("Found %d test files" % len(test_files))            for f in test_files[:3]:                print("  %s" % os.path.basename(f))        else:            print("No test audio found. Run scripts/prepare_test_samples.py first")        HAS_MODEL = True    except Exception as e:        print("Model loading failed:", e)        print("Make sure to unzip model weights:")        print("  cd ../DeepFilterNet-main/models/ && unzip DeepFilterNet3.zip")        HAS_MODEL = Falseelse:    HAS_MODEL = False    print("Skipping inference (DeepFilterNet not available)")

In [ ]:
# Enhance audio and visualize resultsimport numpy as npif HAS_MODEL and len(test_files) > 0:    # Process first test file    audio, meta = load_audio(test_files[0], p.sr, 'cpu')    enhanced = enhance(model, df_state, audio, pad=True)    # Plot waveform comparison    fig, axes = plt.subplots(2, 1, figsize=(14, 5))    t_axis = np.arange(len(audio[0])) / p.sr    axes[0].plot(t_axis, audio[0].numpy())    axes[0].set_title("Noisy Input: %s" % os.path.basename(test_files[0]))    axes[0].set_xlabel("Time (s)")    t_axis_enh = np.arange(enhanced.shape[-1]) / p.sr    axes[1].plot(t_axis_enh, enhanced[0].cpu().numpy())    axes[1].set_title("Enhanced Output")    axes[1].set_xlabel("Time (s)")    plt.tight_layout()    plt.show()    # Spectrogram comparison    import torch    noisy_spec = torch.stft(audio[0], p.fft_size, p.hop_size, return_complex=True)    enh_spec = torch.stft(enhanced[0].cpu(), p.fft_size, p.hop_size, return_complex=True)    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    axes[0].imshow(20*np.log10(noisy_spec.abs().numpy() + 1e-8),                   aspect='auto', origin='lower', cmap='magma')    axes[0].set_title("Noisy Spectrogram")    axes[1].imshow(20*np.log10(enh_spec.abs().numpy() + 1e-8),                   aspect='auto', origin='lower', cmap='magma')    axes[1].set_title("Enhanced Spectrogram")    plt.tight_layout()    plt.show()else:    print("Skipping enhancement visualization")    print("To run this demo:")    print("  1. Unzip model weights: models/DeepFilterNet3.zip")    print("  2. Generate test samples: python scripts/prepare_test_samples.py")    print("  3. Re-run this notebook")

## Section 6: ERB vs Mel Filterbank VisualizationUnderstanding why DeepFilterNet uses ERB instead of Mel:

In [ ]:
# Compare ERB and Mel filterbanksimport numpy as npimport matplotlib.pyplot as pltsr = 48000n_fft = 960freq_bins = n_fft // 2 + 1freqs = np.linspace(0, sr // 2, freq_bins)# Simple Mel filterbankdef mel_filterbank(n_mels, sr, n_fft):    n_freqs = n_fft // 2 + 1    freqs = np.linspace(0, sr // 2, n_freqs)    mel_freqs = 2595 * np.log10(1 + freqs / 700)    mel_points = np.linspace(mel_freqs[0], mel_freqs[-1], n_mels + 2)    hz_points = 700 * (10 ** (mel_points / 2595) - 1)    fb = np.zeros((n_mels, n_freqs))    for i in range(n_mels):        f_left = hz_points[i]        f_center = hz_points[i + 1]        f_right = hz_points[i + 2]        for j in range(n_freqs):            if freqs[j] >= f_left and freqs[j] <= f_center:                fb[i, j] = (freqs[j] - f_left) / (f_center - f_left + 1e-8)            elif freqs[j] > f_center and freqs[j] <= f_right:                fb[i, j] = (f_right - freqs[j]) / (f_right - f_center + 1e-8)    return fb# Simple ERB filterbank (approximate)def erb_filterbank(n_erbs, sr, n_fft):    n_freqs = n_fft // 2 + 1    freqs = np.linspace(0, sr // 2, n_freqs)    # ERB-rate scale: ERB_N(f) = 21.4 * log10(0.00437*f + 1)    erb_points = 21.4 * np.log10(0.00437 * freqs + 1)    erb_grid = np.linspace(erb_points[0], erb_points[-1], n_erbs + 2)    hz_grid = (10 ** (erb_grid / 21.4) - 1) / 0.00437    fb = np.zeros((n_erbs, n_freqs))    for i in range(n_erbs):        f_left = hz_grid[i]        f_center = hz_grid[i + 1]        f_right = hz_grid[i + 2]        for j in range(n_freqs):            if freqs[j] >= f_left and freqs[j] <= f_center:                fb[i, j] = (freqs[j] - f_left) / (f_center - f_left + 1e-8)            elif freqs[j] > f_center and freqs[j] <= f_right:                fb[i, j] = (f_right - freqs[j]) / (f_right - f_center + 1e-8)    return fbmel_fb = mel_filterbank(32, sr, n_fft)erb_fb = erb_filterbank(32, sr, n_fft)fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].imshow(mel_fb, aspect='auto', origin='lower', cmap='hot',               extent=[0, sr//2, 0, 32])axes[0].set_title("Mel Filterbank (32 bands)")axes[0].set_xlabel("Frequency (Hz)")axes[0].set_ylabel("Band index")axes[1].imshow(erb_fb, aspect='auto', origin='lower', cmap='hot',               extent=[0, sr//2, 0, 32])axes[1].set_title("ERB Filterbank (32 bands)")axes[1].set_xlabel("Frequency (Hz)")axes[1].set_ylabel("Band index")plt.tight_layout()plt.show()print("Observation: ERB filterbank allocates more bands to low frequencies")print("This matches the human cochlear frequency resolution pattern")print("CI electrodes follow a similar non-linear frequency mapping")

## SummaryThis session covered:1. DeepFilterNet code structure and organization2. Configuration parameters and their physical meaning3. Model architecture with detailed data flow tracking4. Comparison with DeepACE architecture5. Running inference with pretrained models6. ERB vs Mel filterbank visualization**Key Takeaways:**- DeepFilterNet uses a two-stage approach: mask-based enhancement + deep filtering- ERB domain is chosen for its connection to human auditory perception- The encoder-decoder pattern with skip connections is shared with DeepACE- Real-time inference is possible with ~5ms latency**Homework:**- Read `loss.py` to understand the multi-component loss function- Try enhancing your own audio files- Prepare for next session: think about how to connect DeepFilterNet with ACE